# Anhedonic AI — Condition Comparison
Compares **normal** vs **anhedonic** vs **neutral** conditions across:
1. Behavioral results (point choices)
2. Residual stream activations (layer-wise)
3. MLP neuron activations (layer-wise)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
# Behavioral results
df_normal    = pd.read_csv('results_normal_mode.csv')
df_anhedonic = pd.read_csv('results_anhedonic_mode.csv')
df_neutral   = pd.read_csv('results_neutral_mode.csv')

# Activations — shape: (100, 28, dim)
res_normal    = np.load('activations/activations_residual_normal.npy')    # or adjust path
res_anhedonic = np.load('activations/activations_residual_anhedonic.npy')
res_neutral   = np.load('activations/activations_residual_neutral.npy')

neu_normal    = np.load('activations/activations_neurons_normal.npy')
neu_anhedonic = np.load('activations/activations_neurons_anhedonic.npy')
neu_neutral   = np.load('activations/activations_neurons_neutral.npy')

n_layers = res_normal.shape[1]
print(f'Residual shape : {res_normal.shape}  →  (prompts, layers, hidden_dim)')
print(f'Neuron shape   : {neu_normal.shape}  →  (prompts, layers, ffn_dim)')
print(f'Layers: {n_layers}')

## 2. Behavioral Results

In [ ]:
point_values = [1, 10, 50, 100]
conditions   = ['Normal', 'Anhedonic']
dfs          = [df_normal, df_anhedonic]
colors       = ['#2196F3', '#F44336']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Point distribution bar chart ---
ax = axes[0]
x  = np.arange(len(point_values))
w  = 0.35
for i, (label, df, color) in enumerate(zip(conditions, dfs, colors)):
    counts = [((df['chosen_points'] == p).sum() / len(df)) * 100 for p in point_values]
    ax.bar(x + i*w, counts, w, label=label, color=color, alpha=0.85)
ax.set_xticks(x + w/2)
ax.set_xticklabels([f'{p} pts' for p in point_values])
ax.set_ylabel('% of choices')
ax.set_title('Point Choice Distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# --- Mean points chosen ---
ax = axes[1]
means = [df['chosen_points'].mean() for df in dfs]
sems  = [df['chosen_points'].sem() for df in dfs]
bars  = ax.bar(conditions, means, color=colors, alpha=0.85, yerr=sems, capsize=5)
ax.set_ylabel('Mean points chosen')
ax.set_title('Mean Points Chosen (± SEM)')
ax.set_ylim(0, 110)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 3, f'{mean:.1f}', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Significance test
t_stat, p_val = stats.ttest_ind(df_normal['chosen_points'].dropna(), df_anhedonic['chosen_points'].dropna())
ax.set_xlabel(f't={t_stat:.2f}, p={p_val:.4f}')

# --- High reward (100pt) choice rate ---
ax = axes[2]
rates = [(df['chosen_points'] == 100).mean() * 100 for df in dfs]
bars  = ax.bar(conditions, rates, color=colors, alpha=0.85)
ax.set_ylabel('% choosing 100 pts')
ax.set_title('High-Reward Choice Rate (100 pts)')
ax.set_ylim(0, 100)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, rate + 2, f'{rate:.1f}%', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Behavioral Results: Normal vs Anhedonic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_behavioral.png', bbox_inches='tight')
plt.show()
print(f'Normal mean: {means[0]:.1f} | Anhedonic mean: {means[1]:.1f} | Δ = {means[0]-means[1]:.1f} pts')

## 3. Residual Stream — Layer-wise L2 Distance

In [ ]:
# Mean L2 distance between conditions at each layer
def mean_l2(a, b):
    """Mean L2 distance per layer. a, b: (n, layers, dim)"""
    return np.linalg.norm(a - b, axis=2).mean(axis=0)  # (layers,)

l2_anh_vs_norm = mean_l2(res_anhedonic, res_normal)
l2_neu_vs_norm = mean_l2(res_neutral,   res_normal)
l2_anh_vs_neu  = mean_l2(res_anhedonic, res_neutral)

layers = np.arange(n_layers)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(layers, l2_anh_vs_norm, 'r-o', ms=4, label='Anhedonic vs Normal',  linewidth=2)
ax.plot(layers, l2_neu_vs_norm, 'b-o', ms=4, label='Neutral vs Normal',    linewidth=2)
ax.plot(layers, l2_anh_vs_neu,  'g-o', ms=4, label='Anhedonic vs Neutral', linewidth=2)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean L2 Distance')
ax.set_title('Residual Stream — Layer-wise L2 Distance Between Conditions')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(layers)
plt.tight_layout()
plt.savefig('fig_residual_l2.png', bbox_inches='tight')
plt.show()

## 4. Residual Stream — Layer-wise Cosine Similarity

In [ ]:
def mean_cosine(a, b):
    """Mean cosine similarity per layer."""
    sims = []
    for l in range(a.shape[1]):
        al = a[:, l, :]  # (n, dim)
        bl = b[:, l, :]
        dot  = (al * bl).sum(axis=1)
        norm = np.linalg.norm(al, axis=1) * np.linalg.norm(bl, axis=1)
        sims.append((dot / (norm + 1e-8)).mean())
    return np.array(sims)

cos_anh_vs_norm = mean_cosine(res_anhedonic, res_normal)
cos_neu_vs_norm = mean_cosine(res_neutral,   res_normal)
cos_anh_vs_neu  = mean_cosine(res_anhedonic, res_neutral)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(layers, cos_anh_vs_norm, 'r-o', ms=4, label='Anhedonic vs Normal',  linewidth=2)
ax.plot(layers, cos_neu_vs_norm, 'b-o', ms=4, label='Neutral vs Normal',    linewidth=2)
ax.plot(layers, cos_anh_vs_neu,  'g-o', ms=4, label='Anhedonic vs Neutral', linewidth=2)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean Cosine Similarity')
ax.set_title('Residual Stream — Layer-wise Cosine Similarity Between Conditions')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(layers)
plt.tight_layout()
plt.savefig('fig_residual_cosine.png', bbox_inches='tight')
plt.show()

## 5. MLP Neurons — Mean Activation per Layer

In [ ]:
# Mean absolute activation per layer
mean_act_normal    = np.abs(neu_normal).mean(axis=(0, 2))     # (layers,)
mean_act_anhedonic = np.abs(neu_anhedonic).mean(axis=(0, 2))
mean_act_neutral   = np.abs(neu_neutral).mean(axis=(0, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(layers, mean_act_normal,    'b-o', ms=4, label='Normal',    linewidth=2)
ax.plot(layers, mean_act_anhedonic, 'r-o', ms=4, label='Anhedonic', linewidth=2)
ax.plot(layers, mean_act_neutral,   'g-o', ms=4, label='Neutral',   linewidth=2)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean |Activation|')
ax.set_title('MLP Neuron Mean Activation per Layer')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(layers)

# Difference: anhedonic - normal
ax = axes[1]
diff = mean_act_anhedonic - mean_act_normal
colors_bar = ['#F44336' if d > 0 else '#2196F3' for d in diff]
ax.bar(layers, diff, color=colors_bar, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Δ Mean |Activation| (Anhedonic − Normal)')
ax.set_title('MLP Activation Change: Anhedonic − Normal')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(layers)

plt.suptitle('MLP Neuron Activations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_neuron_activations.png', bbox_inches='tight')
plt.show()

## 6. Top Neurons Most Changed by Anhedonic Condition

In [ ]:
TOP_N = 20

# Mean activation per neuron per layer: (layers, ffn_dim)
mean_neu_normal    = neu_normal.mean(axis=0)     # (28, 18944)
mean_neu_anhedonic = neu_anhedonic.mean(axis=0)

# Absolute difference across all layers flattened
diff_matrix = np.abs(mean_neu_anhedonic - mean_neu_normal)  # (28, 18944)

# Find top neurons globally (layer, neuron_idx)
flat_idx    = np.argsort(diff_matrix.flatten())[::-1][:TOP_N]
top_layers  = flat_idx // diff_matrix.shape[1]
top_neurons = flat_idx %  diff_matrix.shape[1]
top_diffs   = diff_matrix.flatten()[flat_idx]

fig, ax = plt.subplots(figsize=(12, 6))
labels = [f'L{l}-N{n}' for l, n in zip(top_layers, top_neurons)]
bar_colors = ['#F44336' if (mean_neu_anhedonic[l, n] - mean_neu_normal[l, n]) > 0 
              else '#2196F3' for l, n in zip(top_layers, top_neurons)]
ax.bar(range(TOP_N), top_diffs, color=bar_colors, alpha=0.85)
ax.set_xticks(range(TOP_N))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('|Δ Mean Activation| (Anhedonic − Normal)')
ax.set_title(f'Top {TOP_N} Neurons Most Changed by Anhedonic Condition\n(red=increased, blue=decreased)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_top_neurons.png', bbox_inches='tight')
plt.show()

print('Top changed neurons:')
for l, n, d in zip(top_layers, top_neurons, top_diffs):
    direction = '↑' if mean_neu_anhedonic[l, n] > mean_neu_normal[l, n] else '↓'
    print(f'  Layer {l:2d}, Neuron {n:5d}: Δ={d:.4f} {direction}')

## 7. PCA — Residual Stream Separation Across Conditions

In [ ]:
# Pick a few representative layers to visualize
LAYERS_TO_PLOT = [0, 7, 14, 21, 27]

fig, axes = plt.subplots(1, len(LAYERS_TO_PLOT), figsize=(18, 4))

for ax, layer in zip(axes, LAYERS_TO_PLOT):
    # Stack all three conditions
    X = np.vstack([
        res_normal[:, layer, :],
        res_anhedonic[:, layer, :],
        res_neutral[:, layer, :],
    ])
    labels_pca = ['Normal'] * 100 + ['Anhedonic'] * 100 + ['Neutral'] * 100
    colors_pca = ['#2196F3'] * 100 + ['#F44336'] * 100 + ['#4CAF50'] * 100

    pca = PCA(n_components=2)
    X2  = pca.fit_transform(StandardScaler().fit_transform(X))

    for color, label in [('#2196F3', 'Normal'), ('#F44336', 'Anhedonic'), ('#4CAF50', 'Neutral')]:
        mask = np.array(labels_pca) == label
        ax.scatter(X2[mask, 0], X2[mask, 1], c=color, label=label, alpha=0.6, s=20)

    var = pca.explained_variance_ratio_
    ax.set_title(f'Layer {layer}\n(PC1={var[0]:.1%}, PC2={var[1]:.1%})')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    if layer == LAYERS_TO_PLOT[0]:
        ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.suptitle('PCA of Residual Stream — Condition Separation by Layer', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_pca_residual.png', bbox_inches='tight')
plt.show()

## 8. Neuron Activation Heatmap — All Layers

In [ ]:
# Heatmap of mean neuron activation difference (anhedonic - normal) across layers
# Subsample neurons for readability
N_NEURONS_SAMPLE = 500
rng = np.random.default_rng(42)
sampled_neurons = rng.choice(neu_normal.shape[2], N_NEURONS_SAMPLE, replace=False)

diff_sampled = (mean_neu_anhedonic - mean_neu_normal)[:, sampled_neurons]  # (28, 500)

fig, ax = plt.subplots(figsize=(16, 6))
vmax = np.percentile(np.abs(diff_sampled), 99)
im = ax.imshow(diff_sampled, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xlabel(f'Neuron index (sampled {N_NEURONS_SAMPLE}/{neu_normal.shape[2]})')
ax.set_ylabel('Layer')
ax.set_title('MLP Neuron Activation Difference (Anhedonic − Normal)\nRed = increased, Blue = decreased')
ax.set_yticks(range(n_layers))
plt.colorbar(im, ax=ax, label='Δ Mean Activation')
plt.tight_layout()
plt.savefig('fig_neuron_heatmap.png', bbox_inches='tight')
plt.show()

## 9. Summary Statistics

In [ ]:
print('='*50)
print('BEHAVIORAL SUMMARY')
print('='*50)
for label, df in [('Normal', df_normal), ('Anhedonic', df_anhedonic), ('Neutral', df_neutral)]:
    pts = df['chosen_points'].dropna() if 'chosen_points' in df.columns else pd.Series([])
    if len(pts):
        print(f'\n{label}:')
        print(f'  Mean points : {pts.mean():.1f}')
        print(f'  Std         : {pts.std():.1f}')
        print(f'  100pt rate  : {(pts==100).mean()*100:.1f}%')
        print(f'  1pt rate    : {(pts==1).mean()*100:.1f}%')

t_stat, p_val = stats.ttest_ind(
    df_normal['chosen_points'].dropna(),
    df_anhedonic['chosen_points'].dropna()
)
print(f'\nNormal vs Anhedonic t-test: t={t_stat:.3f}, p={p_val:.4f}')

print('\n' + '='*50)
print('ACTIVATION SUMMARY')
print('='*50)
print(f'\nMax L2 shift (residual, anhedonic vs normal): layer {np.argmax(l2_anh_vs_norm)}, Δ={np.max(l2_anh_vs_norm):.4f}')
print(f'Max L2 shift (residual, neutral vs normal):   layer {np.argmax(l2_neu_vs_norm)}, Δ={np.max(l2_neu_vs_norm):.4f}')
print(f'\nMost changed neuron (anhedonic vs normal):')
print(f'  Layer {top_layers[0]}, Neuron {top_neurons[0]}, Δ={top_diffs[0]:.4f}')